### Tools

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("why do parrots talk?")
response.content

'Actually, parrots don’t "talk" in the human sense. They are **mimics**. They learn to reproduce sounds they hear, including human speech, because of a unique combination of biological, social, and evolutionary factors.\n\nHere’s why parrots mimic speech:\n\n### 1. **Specialized Anatomy for Vocalization**\nParrots have a syrinx (voice box) at the base of their trachea that allows them to produce a wide range of sounds. Their tongues are also highly mobile, enabling precise control over airflow and sound shaping, much like humans. This anatomical setup makes them exceptionally good at imitating complex sounds.\n\n### 2. **Social Bonding and Communication**\nIn the wild, parrots are highly social animals that live in flocks. They use vocalizations to:\n- Maintain contact with the flock.\n- Signal danger or food sources.\n- Strengthen pair bonds.\n\nWhen kept as pets, parrots view their human caregivers as part of their "flock." Mimicking human speech is a way to:\n- **Bond** with their h

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response.content)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


Tool: get_weather
Args: {'location': 'New York'}


#### Tool Execution Loop

In [4]:
# Step 1: model generates tool calls
messages = [{"role": "user", "content": "What's the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


It's sunny in New York!


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in New York?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 's52dqq3ak', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_time': 0.075157666, 'completion_tokens_details': None, 'prompt_time': 0.020646125, 'prompt_tokens_details': None, 'queue_time': 0.020598032, 'total_time': 0.095803791}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16890-37df-7025-907e-ec621c36d29c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 's52dqq3ak', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 27, 'total_tokens': 305}),
 ToolMessage(content="It's 